# Wellbore Geology Prediction — Full Resolution
XGBoost on all 773 wells (every row) with Google Drive model caching.
Saves trained model to Drive so you only train once.

**Before running:**
1. Get Kaggle API token: https://www.kaggle.com/settings → Create New API Token
2. (Optional) Connect Google Drive for model caching

In [ ]:
# Mount Google Drive for model caching
from google.colab import drive
drive.mount('/content/drive')

import os
CACHE_DIR = '/content/drive/MyDrive/wellbore_model'
os.makedirs(CACHE_DIR, exist_ok=True)
MODEL_PATH = os.path.join(CACHE_DIR, 'xgb_model.json')
FEATURES_PATH = os.path.join(CACHE_DIR, 'feature_names.txt')

In [ ]:
# Authenticate with Kaggle & download data
from google.colab import files

if not os.path.exists('/content/rogii-wellbore-geology-prediction/train'):
    if not os.path.exists(os.path.expanduser('~/.kaggle/kaggle.json')):
        print("Upload kaggle.json")
        uploaded = files.upload()
        !mkdir -p ~/.kaggle
        !mv kaggle.json ~/.kaggle/
        !chmod 600 ~/.kaggle/kaggle.json

    !kaggle competitions download rogii-wellbore-geology-prediction
    !unzip -q rogii-wellbore-geology-prediction.zip -d /content/rogii-wellbore-geology-prediction
    !rm rogii-wellbore-geology-prediction.zip
    print('Data ready')
else:
    print('Data already downloaded')

In [ ]:
# Install dependencies
!pip install -q xgboost scikit-learn pandas numpy scipy matplotlib

In [ ]:
# Imports
import sys, gc, json, numpy as np, pandas as pd, xgboost as xgb
from pathlib import Path

sys.path.insert(0, '/content/rogii-wellbore-geology-prediction')
from src.data.loader import load_all_wells, load_horizontal, load_typewell
from src.features.build_features import build_features, FEATURE_COLS

TEST_DIR = Path('/content/rogii-wellbore-geology-prediction/test')
OUTPUT = '/content/submission.csv'

In [ ]:
# Load or train model
if os.path.exists(MODEL_PATH):
    print('Loading cached model from Drive...')
    model = xgb.XGBRegressor()
    model.load_model(MODEL_PATH)
    with open(FEATURES_PATH) as f:
        avail = f.read().strip().split(',')
    print(f'Loaded model (features: {len(avail)})')
else:
    print('Training model from scratch...')
    wells = load_all_wells(Path('/content/rogii-wellbore-geology-prediction'), is_train=True)
    all_ids = list(wells.keys())

    X_list, y_list = [], []
    for wid in all_ids:
        hw, tw = wells[wid]
        df = build_features(hw, tw, is_train=True)
        avail = [c for c in FEATURE_COLS if c in df.columns]
        X_list.append(df[avail].values.astype(np.float32))
        y_list.append(df['TVT'].values.astype(np.float32))
        del df, hw, tw; gc.collect()

    X = np.concatenate(X_list, axis=0)
    y = np.concatenate(y_list)
    print(f'Training on {X.shape[0]} rows, {X.shape[1]} features')

    model = xgb.XGBRegressor(
        n_estimators=500, max_depth=7, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.7, min_child_weight=5,
        reg_alpha=0.1, reg_lambda=2.0, random_state=42, verbosity=1,
    )
    model.fit(X, y)

    # Cache to Drive
    model.save_model(MODEL_PATH)
    with open(FEATURES_PATH, 'w') as f:
        f.write(','.join(avail))
    print(f'Model saved to {MODEL_PATH}')

In [ ]:
# Predict test set
print('Predicting...')
well_ids = sorted(set(p.stem.split('__')[0] for p in TEST_DIR.glob('*__horizontal_well.csv')))
all_out_ids, all_tvts = [], []

for wid in well_ids:
    hw = load_horizontal(TEST_DIR, wid, is_train=False)
    tw = load_typewell(TEST_DIR, wid, is_train=False)
    nan_mask = hw['TVT_input'].isna().values
    df = build_features(hw, tw, is_train=False)
    avail_cols = [c for c in FEATURE_COLS if c in df.columns]
    preds = model.predict(df[avail_cols].values.astype(np.float32))
    for idx in np.where(nan_mask)[0]:
        all_out_ids.append(f'{wid}_{idx}')
        all_tvts.append(preds[idx])
    print(f'  {wid}: done')
    del hw, tw, df, preds; gc.collect()

sub = pd.DataFrame({'id': all_out_ids, 'tvt': all_tvts})
sub.to_csv(OUTPUT, index=False)
print(f'Saved {len(sub)} predictions')

from google.colab import files
files.download(OUTPUT)